In [1]:
import parcels
import geopandas as gpd 
import sys
import sys
from pathlib import Path
import xarray as xr
from datetime import timedelta
# go to project root (parent of Parcles/)
sys.path.insert(0, str(Path.cwd().parent))
from functions.funcs import *
import functions.plotting
from importlib import reload 
reload(functions.plotting)
import numpy as np
import tomllib 

## Notebook to test Static field of Forecasts 

1) First method is testing new Persistence Kernal for advection 
2) Second method is standard advection of OSCAR

In [ ]:
def persistence_AdvectionRK4(particle, fieldset, time):  # pragma: no cover
    """Advection of particles using fourth-order Runge-Kutta integration.
    with an added persitence model """
    import numpy as np 
    (u1, v1) = fieldset.UV[particle]
    lon1, lat1 = (particle.lon + u1 * 0.5 * particle.dt, particle.lat + v1 * 0.5 * particle.dt)
    (u2, v2) = fieldset.UV[time + 0.5 * particle.dt, particle.depth, lat1, lon1, particle]
    lon2, lat2 = (particle.lon + u2 * 0.5 * particle.dt, particle.lat + v2 * 0.5 * particle.dt)
    (u3, v3) = fieldset.UV[time + 0.5 * particle.dt, particle.depth, lat2, lon2, particle]
    lon3, lat3 = (particle.lon + u3 * particle.dt, particle.lat + v3 * particle.dt)
    (u4, v4) = fieldset.UV[time + particle.dt, particle.depth, lat3, lon3, particle]
    advection_dlon = (u1 + 2 * u2 + 2 * u3 + u4) / 6.0 * particle.dt  # noqa
    advection_dlat = (v1 + 2 * v2 + 2 * v3 + v4) / 6.0 * particle.dt  # noqa

    ## Calculating persistence 
    persistence_dlon = particle.ui*particle.dt
    persistence_dlat = particle.vi*particle.dt

    # Weighting how much persistence to use
    persistence_frac = np.exp(-particle.age/particle.tau)
    if particle.age < 4*particle.tau: 
        #print(particle.dt, particle.ui, persistence_frac)
        persistence_frac = np.exp(-particle.age/particle.tau)
    else: 
        persistence_frac = 0

    # final displacement 
    particle_dlon += persistence_dlon*persistence_frac + advection_dlon*(1- persistence_frac)
    particle_dlat += persistence_dlat*persistence_frac + advection_dlat*(1- persistence_frac)

### test 1

In [ ]:
ds = gpd.read_parquet(r"..\Data\Mapped_SAT_MI_Cleanedspeeds.parquet")

target_date = pd.to_datetime("2024-5-12", format="%Y-%m-%d") ## picks dFAD locations one day after this date 

ds_active = querry_date(ds, date = target_date) ## All of the active dFADs at this time 
ds_active = ds_active.reset_index()
columns = ["TimeStamp", "x_speed", "y_speed"]
ds_locations = pd.DataFrame()
for label in columns: 
    longlist, ids = Column_to_List(ds_active, label, idlist = True)
     
    ds_locations[label] = longlist
lat, lon  = list_of_latlon(ds_active, droplast= False)
ds_locations["lat"] = lat
ds_locations["lon"] =lon
ds_locations["BuoyName"] = ids
ds_locations.TimeStamp = pd.to_datetime(ds_locations.TimeStamp)
ds_locations["x_speed_prev"] = (ds_locations.groupby("BuoyName")["x_speed"].transform(lambda x: x.rolling(window=2, min_periods=1).mean()))
ds_locations["y_speed_prev"] = (ds_locations.groupby("BuoyName")["y_speed"].transform(lambda x: x.rolling(window=2, min_periods=1).mean()))

##Filter Timestep by certain threshhold to get locations of FADS within closes  
## UPDATE:This might be better to interp these onto the specific time. 
hourlim = 24
time_threshhold  = pd.Timedelta(hours= hourlim)
time_upper  = target_date + time_threshhold ## This is set for dFADs one day after the date 
time_lower = target_date 
ds_locations = ds_locations.query(f"TimeStamp > @time_lower")
ds_locations = ds_locations.query(f"TimeStamp < @time_upper")
print(f"Amount of sampled dFAD within {hourlim} hrs : {len(ds_locations)}")
ds_locations = ds_locations.drop_duplicates(subset=["lat"], keep="first")
ds_locations = ds_locations.drop_duplicates(subset=["lon"], keep="first").reset_index(drop = True)

## New get only the first point of the day for the forcast.

dFADs = ds_locations.sort_values('TimeStamp').groupby("BuoyName").first()
print(f"Number of Unique dFADs/ points avalable: {len(dFADs)}")
dFADs = dFADs.reset_index()

In [ ]:
name = 'oscar'

if name == 'oscar':
    oscar = xr.open_dataset(rf"..\Data\OSCAR_combined_2021_2025v2.nc")
    oscar = oscar.rename({'lon':'longitude', 'lat':'latitude', 'u' : 'uo', 'v':'vo' })
    oscar = oscar.set_index(longitude='longitude', latitude='latitude')
    oscar = oscar.transpose('time' ,'latitude', 'longitude')
    field_t = oscar.sel(time = target_date, method = "nearest").drop_vars("time") 

if name == 'cmems':
    cmems = xr.open_dataset(rf"..\Data\cmems.nc")
    field_t = cmems.sel(time = target_date, depth = 15, method = "nearest").drop_vars("time")

## Make the model... 
variables  = {"U": "uo", "V": "vo"}
dimensions = {"lat": "latitude", "lon": "longitude"}
runtime = pd.Timedelta(days =7)

# fieldsetperm = parcels.FieldSet.from_netcdf(filenames, variables, dimensions)
fieldset  = parcels.FieldSet.from_xarray_dataset(field_t, variables, dimensions, allow_time_extrapolation= True) 
fieldset.add_constant("halo_west", fieldset.U.grid.lon[0])
fieldset.add_constant("halo_east", fieldset.U.grid.lon[-1])
fieldset.add_constant("halo_north", fieldset.U.grid.lat[-1])
fieldset.add_constant("halo_south", fieldset.U.grid.lat[0])
fieldset.add_periodic_halo(zonal = True , meridional= True)

def boundryCondition(particle, fieldset,time):
    if particle.lon < fieldset.halo_west or particle.lon > fieldset.halo_east:
        particle.delete()
    if particle.lat < fieldset.halo_south or particle.lat > fieldset.halo_north:
        particle.delete()
        
def Age(particle, fieldset, time):
    particle.age += particle.dt / 3600
    
dFADs["timedelta"] = (dFADs.TimeStamp - target_date).dt.total_seconds()

Particles = parcels.ScipyParticle.add_variable("age", initial = 0) 
Particles = Particles.add_variable("Buoyindex", to_write = 'once')
Particles = Particles.add_variable("ui", to_write = 'once' )
Particles = Particles.add_variable("vi", to_write = 'once')
Particles = Particles.add_variable("tau",initial = 0.83*24,to_write = 'once')

pset = parcels.ParticleSet.from_list(fieldset, pclass = Particles , lon = dFADs.lon.to_list(), 
                                     lat = dFADs.lat.to_list() , time = dFADs.timedelta, Buoyindex = dFADs.BuoyName.index, 
                                     ui = dFADs.x_speed_prev/1000/111, vi = dFADs.y_speed_prev/1000/111) 

output_file = pset.ParticleFile(name = "TestParticleFile.zarr", outputdt =timedelta(hours= 1))

pset.execute([parcels.AdvectionRK4, boundryCondition, Age], 
             runtime = timedelta(days = 7), ##this should be 8 days 
             dt = timedelta(minutes =5), 
             output_file = output_file, 
              )


### Test 2

In [ ]:
# First load dFADs active that day
ds = gpd.read_parquet(r"..\Data\Mapped_SAT_MI_Cleanedspeeds.parquet")

target_date = pd.to_datetime("2024-5-12", format="%Y-%m-%d") ## picks dFAD locations one day after this date 

ds_active = querry_date(ds, date = target_date) ## All of the active dFADs at this time 
ds_active = ds_active.reset_index()
columns = ["TimeStamp", "x_speed", "y_speed"]
ds_locations = pd.DataFrame()
for label in columns: 
    longlist, ids = Column_to_List(ds_active, label, idlist = True)
     
    ds_locations[label] = longlist
lat, lon  = list_of_latlon(ds_active, droplast= False)
ds_locations["lat"] = lat
ds_locations["lon"] =lon
ds_locations["BuoyName"] = ids
ds_locations.TimeStamp = pd.to_datetime(ds_locations.TimeStamp)
ds_locations["x_speed_prev"] = (ds_locations.groupby("BuoyName")["x_speed"].transform(lambda x: x.rolling(window=2, min_periods=1).mean()))
ds_locations["y_speed_prev"] = (ds_locations.groupby("BuoyName")["y_speed"].transform(lambda x: x.rolling(window=2, min_periods=1).mean()))

##Filter Timestep by certain threshhold to get locations of FADS within closes  
## UPDATE:This might be better to interp these onto the specific time. 
hourlim = 24
time_threshhold  = pd.Timedelta(hours= hourlim)
time_upper  = target_date + time_threshhold ## This is set for dFADs one day after the date 
time_lower = target_date 
ds_locations = ds_locations.query(f"TimeStamp > @time_lower")
ds_locations = ds_locations.query(f"TimeStamp < @time_upper")
print(f"Amount of sampled dFAD within {hourlim} hrs : {len(ds_locations)}")
ds_locations = ds_locations.drop_duplicates(subset=["lat"], keep="first")
ds_locations = ds_locations.drop_duplicates(subset=["lon"], keep="first").reset_index(drop = True)

## New get only the first point of the day for the forcast.

dFADs = ds_locations.sort_values('TimeStamp').groupby("BuoyName").first()
print(f"Number of Unique dFADs/ points avalable: {len(dFADs)}")
dFADs = dFADs.reset_index()

In [ ]:
name = 'oscar'
usewinds = True

winds = xr.open_dataset(r"..\Data\ERA5_10m_winds.nc")
winds = winds.rename({"lat" :'latitude', 'lon': 'longitude'})
winds = winds.sortby('latitude')

if name == 'oscar':
    oscar = xr.open_dataset(rf"..\Data\OSCAR_combined_2021_2025v2.nc")
    oscar = oscar.rename({'lon':'longitude', 'lat':'latitude', 'u' : 'uo', 'v':'vo' })
    oscar = oscar.set_index(longitude='longitude', latitude='latitude')
    oscar = oscar.transpose('time' ,'latitude', 'longitude')
    if usewinds == True:
        windsi = winds.interp_like(oscar)
        m = 7.73709253e-01+1.62143540e-01j
        n = 7.85629581e-03+1.45730160e-03j
        Uo = oscar.uo +oscar.vo*1j
        W = windsi.uo +windsi.vo*1j
        Y = m*Uo + n*W
        oscar['uo'] = Y.real
        oscar['vo'] = Y.imag
    field_t = oscar.sel(time = target_date, method = "nearest").drop_vars("time") 

if name == 'cmems':
    cmems = xr.open_dataset(rf"..\Data\cmems.nc")
    if usewinds == True:
        windsi = winds.interp_like(cmems)
        m = 7.73709253e-01+1.62143540e-01j
        n = 7.85629581e-03+1.45730160e-03j
        Uo = cmems.uo +cmems.vo*1j
        W = windsi.uo +windsi.vo*1j
        Y = m*Uo + n*W
        cmems['uo'] = Y.real
        cmems['vo'] = Y.imag
    field_t = cmems.sel(time = target_date, depth = 15, method = "nearest").drop_vars("time")

variables  = {"U": "uo", "V": "vo"}
dimensions = {"lat": "latitude", "lon": "longitude"}
runtime = pd.Timedelta(days =7)

# fieldsetperm = parcels.FieldSet.from_netcdf(filenames, variables, dimensions)
fieldset  = parcels.FieldSet.from_xarray_dataset(field_t, variables, dimensions, allow_time_extrapolation= True) 
fieldset.add_constant("halo_west", fieldset.U.grid.lon[0])
fieldset.add_constant("halo_east", fieldset.U.grid.lon[-1])
fieldset.add_constant("halo_north", fieldset.U.grid.lat[-1])
fieldset.add_constant("halo_south", fieldset.U.grid.lat[0])
fieldset.add_periodic_halo(zonal = True , meridional= True)

def boundryCondition(particle, fieldset,time):
    if particle.lon < fieldset.halo_west or particle.lon > fieldset.halo_east:
        particle.delete()
    if particle.lat < fieldset.halo_south or particle.lat > fieldset.halo_north:
        particle.delete()
        
def Age(particle, fieldset, time):
    particle.age += particle.dt / 3600
    
dFADs["timedelta"] = (dFADs.TimeStamp - target_date).dt.total_seconds()

Particles = parcels.ScipyParticle.add_variable("age", initial = 0) 
Particles = Particles.add_variable("Buoyindex", to_write = 'once')
Particles = Particles.add_variable("ui", to_write = 'once' )
Particles = Particles.add_variable("vi", to_write = 'once')
Particles = Particles.add_variable("tau",initial = 0.83*24,to_write = 'once')
pset = parcels.ParticleSet.from_list(fieldset, pclass = Particles , lon = dFADs.lon.to_list(), 
                                     lat = dFADs.lat.to_list() , time = dFADs.timedelta, Buoyindex = dFADs.BuoyName.index, 
                                     ui = dFADs.x_speed_prev/1000/111, vi = dFADs.y_speed_prev/1000/111) 

output_file = pset.ParticleFile(name = "TestParticleFile2.zarr", outputdt =timedelta(hours= 1))
pset.execute([persistence_AdvectionRK4, boundryCondition, Age], 
             runtime = timedelta(days = 7), ##this should be 8 days 
             dt = timedelta(minutes =5), 
             output_file = output_file, )

In [ ]:
## getting True data
def Forcast_snippit(ds: gpd.GeoDataFrame, dates, startdate, length)-> gpd.GeoDataFrame: 
    """Grad only the snipbit of dFAD trajectory that lines up with forcast window"""
    ds_s = ds.copy()
    forecast_end = startdate + length
    for i in range(len(ds)): ## Try and grab at the exact start times from dates they should be the same
        timelist = (ds_s.at[i,"TimeStamp"])
        timelist = pd.to_datetime(timelist)
        mask = (timelist >= dates[i]) & (timelist <= forecast_end)
        timelist = timelist[mask]
        coords = np.asarray(ds.at[i,"geometry"].coords)
        filtered_coords = coords[mask]
        ds_s.at[i,"TimeStamp"] = timelist
        if len(filtered_coords) > 1:
             ds_s.at[i,"geometry"] = sp.geometry.LineString(filtered_coords)
        else: 
            ds_s.at[i,"geometry"] = None
    return ds_s

buoy_list = dFADs.BuoyName.tolist()
ds_filtered = ds_active[ds_active["BuoyName"].isin(buoy_list)].reset_index(drop = True)

ds_short_t = Forcast_snippit(ds_filtered, dFADs.TimeStamp, target_date, pd.Timedelta(days = 7))
one_dfad  = ds_short_t.query('BuoyName == "M3i+136927"').reset_index(drop = True)

In [ ]:
## plotting two tests 
ds = xr.open_zarr("TestParticleFile.zarr", decode_timedelta=True)
ds2 = xr.open_zarr("TestParticleFile2.zarr", decode_timedelta=True)
fig, ax = plt.subplots()
for i in range(len(ds_filtered)):
    ax = functions.plotting.OneTrajectory(ds_short_t, i, ax)

#ax = functions.plotting.OneTrajectory(one_dfad, 0, ax, label = "test", color = "red")
ax.plot(ds.lon.T, ds.lat.T, "-", color="green", label="oscar")
ax.plot(ds2.lon.T, ds2.lat.T, "-", color="k", label="persistence+wind")
plt.scatter(dFADs.lon, dFADs.lat)
ax.set_xlabel("Zonal distance [m]")
ax.set_ylabel("Meridional distance [m]")
ax.legend(handles=ax.get_legend_handles_labels()[0][:2])
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys())
plt.show()

In [ ]:
np.printoptions(supress = True, precision= 3 )
print(ds.age.values[1,:])

In [ ]:
## Procedure to get OSCAR into same format as cmems
oscar = xr.open_dataset(rf"..\Data\OSCAR_combined_2021_2025v2.nc")
oscar = oscar.rename({'lon':'longitude', 'lat':'latitude', 'u' : 'uo', 'v':'vo' })
oscar = oscar.set_index(longitude='longitude', latitude='latitude')
oscar = oscar.transpose('time', 'latitude', 'longitude')

In [ ]:
## Procedure to get wind into same format as cmems
winds = xr.open_dataset(r"..\Data\ERA5_10m_winds.nc")
winds = winds.rename({"lat" :'latitude', 'lon': 'longitude'})
winds = winds.sortby('latitude')
#winds = winds.rename({'lat': 'latitude' ,'lon' : 'longitude'})
# windsr = winds.rename_vars({"uo":'u', 'vo': 'v'})
windsi = winds.interp_like(oscar)

In [ ]:
oscar = xr.open_dataset(fname)

m = 9.20652565e-01+1.94924156e-02j
n = 1.05991333e-02+7.39510759e-03j
Uo = oscar.uo +oscar.vo*1j
W = windsi.uo +windsi.vo*1j
Y = m*Uo + n*W

#### Going from model output to CSV Needed
- ideally this is going to be stored in memory and only write when the CSV is full

In [ ]:
output = xr.open_zarr("TestParticleFile.zarr")
display(output)

In [ ]:
# print(output.time[2,:].values/1e9)
# print(output.age[2,:].values*3600)

tage = output.age[10,:].values

tage

In [ ]:
output["date"] = (["trajectory", "obs"],output.time.values + target_date)
output["leadtime"] = (["trajectory", "obs"],(output.time[:,:] - output.time[:,0]).values)
output["leadtimeint"] = (["trajectory", "obs"],output.leadtime[:,:].values.astype("int64")/10e9)

In [ ]:
limelist = ds_short_t.iloc[i,:].TimeStamp -  ds_short_t.iloc[i,:].TimeStamp[0]
limelist.astype("int64")/1e9 ### lead time in seconds


In [ ]:
print(len(dFADs))
print(len(ds_short_t))

In [ ]:
dsout = pd.DataFrame(columns = ["BuoyID","Time", "lat_true", "lon_true", "lat_forcast", "lon_forcast", "leadtime"])
masklarge =~ np.isnan(output.lat[:,:].values)
dFADs_s = dFADs.iloc[output.Buoyindex.values]
# ds_short_ts = ds_short_t[ds_short_t["BuoyName"].isin(dFADs_s.BuoyName.values)]
order = dFADs_s.BuoyName.tolist()
order_map = {name: i for i, name in enumerate(order)}
mask = ds_short_t["BuoyName"].isin(order)
ds_short_ts = (
    ds_short_t[mask]
    .assign(order_index=lambda df: df["BuoyName"].map(order_map))
    .sort_values("order_index")
    .drop(columns="order_index")
    .reset_index(drop=True)
)

for i, index in enumerate(output.Buoyindex.values): 
    id = dFADs.BuoyName[index]
    #row = ds_short_t.query("BuoyName == @id") ## could be faster to sort ds_short first then use index to find dFAD
    row=  ds_short_ts.iloc[i]
    Times= row["TimeStamp"]
    starttime = Times[0]
    dFAD_leadtime = (Times - starttime).total_seconds() ## convets leadtime to seconds
    #print((Times - starttime).total_seconds())
    #print(dFAD_leadtime)
    mask = masklarge[i,:] 
    leadtimes = output.age[i,:].values[mask]*3600 ## converts to seconds also. 
    lats = output.lat[i,:].values[mask]
    lons = output.lon[i,:].values[mask]
    lat_interp = np.interp(dFAD_leadtime[1:], leadtimes,lats)
    lon_interp = np.interp(dFAD_leadtime[1:], leadtimes, lons) 
    if row["geometry"] == None:
        continue
    lon_true, lat_true= row["geometry"].xy
    Bouylist = [id]*len(lat_true[1:]) 
    #print(len(Bouylist), len (Times), len(lat_true), len(lon_true), len(lon_interp), len(lat_interp), len(leadtimes))
    dstemp= pd.DataFrame({"BuoyID": Bouylist, "Time": Times[1:], "lat_true": lat_true[1:],"lon_true":lon_true[1:], "lat_forcast": lat_interp, "lon_forcast": lon_interp, "leadtime": dFAD_leadtime[1:]/3600 })
    dsout = pd.concat([dsout,dstemp])


In [ ]:
dsout.iloc[50:150]